In [12]:
import os
import asyncio
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

# 1. Load the file from your validated path
load_dotenv(override=True)

# 2. Setup the Kernel
kernel = Kernel()

# 3. Add your Azure OpenAI Service
# Note: Ensure the variable names match your .env EXACTLY
kernel.add_service(
    AzureChatCompletion(
        deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version="2024-08-01-preview" # Latest version for gpt-4o-mini
    )
)

# 4. Ask a question to verify the model is responding
try:
    response = await kernel.invoke_prompt("Hello!")
    print(response)
except Exception as e:
    # This will print the actual error from Azure (404, 401, etc.)
    print(f"Full Error: {e}")
    if hasattr(e, '__cause__'):
        print(f"Cause: {e.__cause__}")

print(f"Current Directory: {os.getcwd()}")
print(f"File exists: {os.path.exists('.env')}")

Hello! How can I assist you today?
Current Directory: c:\Users\ssivasub\OneDrive - Capgemini\Architecture\AI\PythonSK
File exists: True


In [13]:
import yaml
import os
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

# 1. Load the YAML configuration
yaml_path = os.path.join("config", "agents.yaml")

with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

# 2. Dictionary to hold our initialized agents
agents = {}

# 3. Loop through the YAML and create Agent objects
for agent_data in config["agents"]:
    name = agent_data["name"]
    
    agents[name] = ChatCompletionAgent(
        service=kernel.get_service(type=AzureChatCompletion),
        name=name,
        description=agent_data["description"],
        instructions=agent_data["instructions"]
    )
    print(f"✅ Agent '{name}' initialized from YAML.")

# 4. Create easy references for your Orchestrator (Cell 3)
architect_agent = agents["Architect"]
security_agent = agents["SecurityExpert"]
coder_agent = agents["Coder"]

print("\n--- All Agents Ready for Group Chat ---")


✅ Agent 'Architect' initialized from YAML.
✅ Agent 'SecurityExpert' initialized from YAML.
✅ Agent 'Coder' initialized from YAML.

--- All Agents Ready for Group Chat ---


In [ ]:
from semantic_kernel.agents import AgentGroupChat
from semantic_kernel.contents import ChatMessageContent, AuthorRole
from semantic_kernel.prompt_template import PromptTemplateConfig
from semantic_kernel.functions import KernelArguments
from semantic_kernel.agents.strategies import KernelFunctionTerminationStrategy

# Refined Termination Function
term_func = kernel.add_function(
    plugin_name="Strategy", 
    function_name="Term",
    prompt_template_config=PromptTemplateConfig(
        template="""
        Review the message history below. 
        Determine if the final response contains the exact word 'TERMINATE'.
        Respond with ONLY 'true' or 'false'.
        
        History: {{$_history_}}
        """,
        template_format="semantic-kernel",
        allow_dangerously_set_content=True
    )
)

# Implementation in the Group Chat
termination_strategy = KernelFunctionTerminationStrategy(
    agents=[coder_agent], # Only trigger termination after the Coder speaks
    function=term_func,
    kernel=kernel,
    # Use 'in' and strip whitespace to be robust against " True", "true.", etc.
    result_parser=lambda r: "true" in str(r.value).lower().strip(),
    history_variable_name="_history_",
    maximum_iterations=6 # Safety net to prevent infinite loops
)

from semantic_kernel.agents.strategies import SequentialSelectionStrategy

group_chat = AgentGroupChat(
    agents=[architect_agent, security_agent, coder_agent], # Sequence: Design -> Security -> Code
    selection_strategy=SequentialSelectionStrategy(),
    termination_strategy=termination_strategy # Linked to the Coder agent above
)


# 3. Fast Session Test
async def fast_session():
    user_input = "Build a Python script that encrypts a local text file using AES"
    print(f"Goal: {user_input}\n" + "="*30)
    
    await group_chat.add_chat_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))

    # Invoke the group chat and print responses
    async for response in group_chat.invoke():
        print(f"\n[{response.name.upper()}]:\n{response.content}")
        print("-" * 20)

await fast_session()


Goal: Build a Python script that encrypts a local text file using AES

[ARCHITECT]:
- Use the `Cryptography` library to implement AES encryption, ensuring to include key generation, encryption mode (e.g., CBC), and IV handling.
- Write the encrypted output to a new file, ensuring to handle file reading/writing correctly and securely.
--------------------

[SECURITYEXPERT]:
Security Approved.
--------------------

[CODER]:
Here is a Python script that encrypts a local text file using AES encryption from the `cryptography` library. The script generates a key, encrypts the content of the text file, and saves it to a new file.

```python
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding
import os
import base64

def generate_key():
    return os.urandom(32)  # AES-256

def encrypt_file(input_file, output_file, key):
    # Generate a random initialization vec

In [15]:
import time
from semantic_kernel.contents import ChatHistory
from semantic_kernel.connectors.ai.open_ai import AzureChatPromptExecutionSettings

async def quick_diagnose():
    start_time = time.time()
    print("Sending direct test request to Azure (v1.41.0 format)...")
    try:
        # 1. Get the service
        chat_service = kernel.get_service(type=AzureChatCompletion)
        
        # 2. Prepare the required arguments for v1.41.0
        history = ChatHistory()
        history.add_user_message("Reply with only the word 'Active'.")
        settings = AzureChatPromptExecutionSettings(max_tokens=10)

        # 3. Call with required positional arguments
        response = await chat_service.get_chat_message_contents(
            chat_history=history,
            settings=settings
        )
        
        duration = time.time() - start_time
        print(f"✅ Success! Azure responded in {duration:.2f} seconds.")
        print(f"Response: {response[0].content}")
        
    except Exception as e:
        print(f"❌ Connection Failed: {e}")
        if "401" in str(e): print("Tip: Check your API Key.")
        if "404" in str(e): print("Tip: Check your Deployment Name/Endpoint.")

await quick_diagnose()


Sending direct test request to Azure (v1.41.0 format)...
✅ Success! Azure responded in 5.68 seconds.
Response: Active
